# Analysis workflow tests

Loads the survey and sensor data, builds a panel dataset, and converts it for two kinds of analysis:

- [SurvivalScale](https://github.com/tazzben/Likert-estimation) (section 5), for Likert-scale responses as the dependent variable.
- OLS with [statsmodels](https://www.statsmodels.org/) (section 6), for a continuous dependent variable such as a slider response or a sensor measure.

Each section ends with assertions that fail loudly if something is wrong.

Install the estimators with `pip install SurvivalScale statsmodels` to run the final sections.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
import sys

for project_root in (Path.cwd(), *Path.cwd().parents):
    src_dir = project_root / "src"
    if (src_dir / "CBACABLab").is_dir():
        sys.path.insert(0, str(src_dir))
        break

import CBACABLab



DATABASE = "data.db"
SURVEY_FILE = "Survey.xlsx"

mappingBetweenQuestionIdAndQuestionName = CBACABLab.AutoDetermineQuestionIdtoQuestionNameMapping(SURVEY_FILE, surveyCSVName="Survey.csv")


In [2]:

SENSOR_FOLDER = "./sensorData/"
REBUILD = True  # reload the survey and sensor data; set False to reuse the existing database

# Respondent-level slider and demographic questions
RESPONDENT_QUESTIONS = ["Q4_4", "Q5_4", "Q6", "Q7"]

# Questions with a Likert-style response, and the scale for each.
# This five-point scale matches the labels used by Q1-Q3 in Survey.xlsx.
AGREEMENT_SCALE = {
    "Strongly disagree": 0,
    "Somewhat disagree": 1,
    "Neither agree nor disagree": 2,
    "Somewhat agree": 3,
    "Strongly agree": 4,
}
LIKERT_SCALES = {question: AGREEMENT_SCALE for question in ["Q1", "Q2", "Q3"]}

TAGS = ["left", "right"]
INSTRUMENT_PROPERTIES = ["Attention", "Joy"]

## 1. Load the survey and sensor data

In [3]:
if REBUILD:
    CBACABLab.LoadSurveyData(databaseName=DATABASE, surveyName=SURVEY_FILE)
    CBACABLab.LoadSensorData(
        databaseName=DATABASE,
        sensorFolder=SENSOR_FOLDER,
        boundedboxes=[(0, 960, 0, 1080)],
        tag="left",
        rebuild=True,
        questionIdtoQuestionName=mappingBetweenQuestionIdAndQuestionName,
        agg="var"
    )
    CBACABLab.LoadSensorData(
        databaseName=DATABASE,
        sensorFolder=SENSOR_FOLDER,
        boundedboxes=[(960, 1920, 0, 1080)],
        tag="right",
        questionIdtoQuestionName=mappingBetweenQuestionIdAndQuestionName
    )


Processing sensor files: 100%|██████████| 10/10 [00:02<00:00,  4.00it/s]


In [4]:
# Test: the tables exist and the survey has data
with sqlite3.connect(DATABASE) as con:
    counts = {
        table: con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        for table in ["FullSurvey", "SensorData", "InstrumentData"]
    }
print(counts)
assert counts["FullSurvey"] > 0, "FullSurvey is empty"
if counts["SensorData"] == 0:
    print("WARNING: SensorData is empty. Sensor columns will be NaN; check the sensor folder and respondent ids.")

{'FullSurvey': 100, 'SensorData': 140, 'InstrumentData': 4981}


In [5]:
# Test: every question we plan to use is in the survey
available = set(CBACABLab.ListQuestions(DATABASE))
wanted = set(RESPONDENT_QUESTIONS) | set(LIKERT_SCALES)
missing = wanted - available
assert not missing, f"Questions not found in the survey: {sorted(missing)}"

## 2. Build the panel dataset

In [6]:
long = CBACABLab.LoadSurveyWithSensorDataLong(questions=["Q1", "Q2"], tagFilter=["left"])
long.count()

respondent_id    732
question         732
response         732
tag              732
pupil            732
duration         732
timeontask       732
property         732
value            732
dtype: int64

In [7]:
panel = CBACABLab.CreatePanelData(
    question_level_columns=list(LIKERT_SCALES),
    respondent_level_columns=RESPONDENT_QUESTIONS,
    tagFilter=TAGS,
    instrumentProperties=INSTRUMENT_PROPERTIES,
    databaseName=DATABASE,
)
panel.head()

,respondent_id,question,response,duration_left,duration_right,pupil_left,pupil_right,timeontask_left,timeontask_right,Attention_left,Attention_right,Joy_left,Joy_right,Q4_4,Q5_4,Q6,Q7
0,1,Q1,Strongly disagree,29670.156690,357.075008,0.012128,2.714873,7190.0,7190.0,0.137679,97.839279,1.061353e-04,0.037362,65.198,22.066,Other,Bachelor's Degree
1,1,Q2,Neither agree nor disagree,52469.772563,1916.269888,0.015667,2.833336,7229.0,7229.0,0.400069,98.041817,3.095435e-02,1.596050,65.198,22.066,Other,Bachelor's Degree
2,1,Q3,Strongly agree,100379.613647,694.727733,0.004729,2.836787,4115.0,4115.0,NaN,97.550504,NaN,0.052147,65.198,22.066,Other,Bachelor's Degree
3,2,Q1,Strongly agree,84675.015681,722.940208,0.055725,2.841391,7770.0,7770.0,0.582230,97.696284,9.528766e-07,0.025649,89.207,94.341,White/ Caucasian,Prefer not to say
4,2,Q2,Neither agree nor disagree,287208.381532,398.548022,0.011280,2.741312,3672.0,3672.0,NaN,97.866264,NaN,0.024139,89.207,94.341,White/ Caucasian,Prefer not to say


In [8]:
normalized_panel = CBACABLab.normalizeSensorDataByRespondent(panel, sensor_columns=["duration_right", "duration_left"])

In [9]:
CBACABLab.returnDescriptiveStatsByRespondent(normalized_panel, sensor_columns=["duration_right_normalized", "duration_left_normalized"])

duration_right_normalized                                 \
                                   mean  std count       min       max   
respondent_id                                                            
1                         -7.401487e-17  1.0     3 -0.944037  1.383937   
2                          0.000000e+00  1.0     2 -1.000000  1.000000   
3                          0.000000e+00  1.0     3 -0.745627  1.413501   
4                          2.590520e-16  1.0     3 -0.791868  1.410680   
5                         -3.700743e-16  1.0     3 -1.277678  1.163866   
6                         -9.251859e-17  1.0     3 -1.379881  0.958169   
7                          7.401487e-17  1.0     3 -0.918910  1.390424   
8                          2.775558e-16  1.0     3 -1.172686  1.270891   
9                          0.000000e+00  1.0     3 -1.398537  0.881121   
10                         1.480297e-16  1.0     3 -1.195382  1.252135   

              duration_left_normalized                                 
                                  mean  std count       min       max  
respondent_id                                                          
1                         0.000000e+00  1.0     3 -1.057765  1.341809  
2                         0.000000e+00  1.0     3 -0.891268  1.396545  
3                        -1.480297e-16  1.0     3 -1.081364  1.329978  
4                         3.700743e-17  1.0     3 -0.787397  1.411050  
5                        -1.110223e-16  1.0     3 -1.413919  0.731962  
6                        -7.401487e-17  1.0     3 -1.398063  0.883597  
7                         1.480297e-16  1.0     3 -1.125395  1.304395  
8                         3.700743e-17  1.0     3 -0.852154  1.403510  
9                         0.000000e+00  1.0     3 -0.769365  1.412329  
10                        0.000000e+00  1.0     3 -0.796414  1.410281

In [10]:
# Tests: panel structure
required = {"respondent_id", "question", "response", *RESPONDENT_QUESTIONS}
assert required <= set(panel.columns), f"Missing columns: {required - set(panel.columns)}"
assert set(panel["question"]) == set(LIKERT_SCALES), "Panel contains unexpected questions"
sensor_columns = [c for c in panel.columns if c.startswith(("pupil_", "duration_"))]
if counts["SensorData"] > 0:
    assert sensor_columns and panel[sensor_columns].notna().any().any(), "No sensor data matched the survey questions"

# Strictly one row per respondent, question, tag
assert not panel.duplicated(["respondent_id", "question"]).any(), "More than one row per respondent and question"

## 3. Convert responses to `k` and `bound`

Responses outside a question's scale (for example "Don't know") can't be converted, so they are listed and dropped first.

In [11]:
def labels(scale):
    if isinstance(scale, str):
        scale = dict(CBACABLab.ListLikertScales(show_values=True))[scale]
    return {str(label).strip().lower() for label in scale}

in_scale = pd.Series(
    [str(r).strip().lower() in labels(LIKERT_SCALES[q]) for q, r in zip(panel["question"], panel["response"])],
    index=panel.index,
)
print("Dropping responses outside their scale:")
print(panel.loc[~in_scale, ["question", "response"]].value_counts())
panel_likert = CBACABLab.AddLikertColumns(panel[in_scale], LIKERT_SCALES)
panel_likert.head(10)

Dropping responses outside their scale:
Series([], Name: count, dtype: int64)


,respondent_id,question,response,duration_left,duration_right,pupil_left,pupil_right,timeontask_left,timeontask_right,Attention_left,Attention_right,Joy_left,Joy_right,Q4_4,Q5_4,Q6,Q7,k,bound
0,1,Q1,Strongly disagree,29670.156690,357.075008,0.012128,2.714873,7190.0,7190.0,0.137679,97.839279,1.061353e-04,0.037362,65.198,22.066,Other,Bachelor's Degree,0,4
1,1,Q2,Neither agree nor disagree,52469.772563,1916.269888,0.015667,2.833336,7229.0,7229.0,0.400069,98.041817,3.095435e-02,1.596050,65.198,22.066,Other,Bachelor's Degree,2,4
2,1,Q3,Strongly agree,100379.613647,694.727733,0.004729,2.836787,4115.0,4115.0,NaN,97.550504,NaN,0.052147,65.198,22.066,Other,Bachelor's Degree,4,4
3,2,Q1,Strongly agree,84675.015681,722.940208,0.055725,2.841391,7770.0,7770.0,0.582230,97.696284,9.528766e-07,0.025649,89.207,94.341,White/ Caucasian,Prefer not to say,4,4
4,2,Q2,Neither agree nor disagree,287208.381532,398.548022,0.011280,2.741312,3672.0,3672.0,NaN,97.866264,NaN,0.024139,89.207,94.341,White/ Caucasian,Prefer not to say,2,4
5,2,Q3,Somewhat disagree,118845.726560,NaN,0.004307,NaN,2874.0,2874.0,0.068228,NaN,2.495373e+03,NaN,89.207,94.341,White/ Caucasian,Prefer not to say,1,4
6,3,Q1,Neither agree nor disagree,70509.309465,264.400428,0.045756,2.847198,7918.0,7918.0,0.251827,98.498688,1.661764e+03,0.036503,100.000,2.458,White/ Caucasian,High school diploma / GED,2,4
7,3,Q2,Strongly disagree,125099.904645,438.112677,0.020714,2.532557,4427.0,4427.0,0.300692,98.661983,5.296318e-01,0.024104,100.000,2.458,White/ Caucasian,High school diploma / GED,0,4
8,3,Q3,Somewhat disagree,228583.921579,257.911149,0.005767,2.658208,7529.0,7529.0,0.070891,98.370094,9.619568e-05,0.038205,100.000,2.458,White/ Caucasian,High school diploma / GED,1,4
9,4,Q1,Somewhat agree,99149.828545,402.003738,0.035072,2.658857,8728.0,8728.0,0.000815,NaN,3.332225e-04,NaN,51.432,7.629,Asian/ Asian American,Master's Degree,3,4


## 4. Prepare the data for SurvivalScale

SurvivalScale expects one row per observation with `id`, `block`, `question`, numeric feature columns, `k` and `bound`. Here `block` is the respondent, so errors can be clustered by respondent.

In [12]:
base = panel_likert.reset_index(drop=True)

FEATURES = ["pupil_left"]
survival = pd.DataFrame({
    "id":  base["respondent_id"],
    "block": base["respondent_id"],
    "question": base["question"],
    "pupil_left": pd.to_numeric(base["pupil_left"], errors="coerce"),
    "k": base["k"].astype(int),
    "bound": base["bound"].astype(int),
}).dropna()
survival.head(10)


,id,block,question,pupil_left,k,bound
0,1,1,Q1,0.012128,0,4
1,1,1,Q2,0.015667,2,4
2,1,1,Q3,0.004729,4,4
3,2,2,Q1,0.055725,4,4
4,2,2,Q2,0.011280,2,4
5,2,2,Q3,0.004307,1,4
6,3,3,Q1,0.045756,2,4
7,3,3,Q2,0.020714,0,4
8,3,3,Q3,0.005767,1,4
9,4,4,Q1,0.035072,3,4


## 5. Run the estimator

In [13]:

# If necessary, install the SurvivalScale package. Uncomment the following line to install it in a Jupyter notebook environment.
# %pip install SurvivalScale 


from SurvivalScale import get_results

if get_results is not None:
    cj, beta, metrics = get_results(
        survival,
        bootstrap_iterations=100,
        alpha=0.05,
        columns=FEATURES,
        block_id="block",
    )
    assert cj is not None and beta is not None and metrics is not None
    display(beta)

Performing bootstrap analysis to estimate confidence intervals and p-values...


100%|██████████| 100/100 [00:03<00:00, 32.48it/s]


,beta,marginal,marginal_discrete,lower_bound,upper_bound,p_value
pupil_left,-23.740318,-5.178378,-0.5,-53.561507,-13.592538,0.0


## 6. OLS with statsmodels

OLS needs a continuous outcome. Likert responses are ordinal, so here the continuous slider response `Q4_4` is the dependent variable, with the Likert response `k` and the other slider `Q5_4` as regressors. Question fixed effects control for the three survey items, and standard errors are clustered by respondent because each respondent appears once per question.

In [14]:
# If necessary, install the statsmodels package. Uncomment the following line to install it in a Jupyter notebook environment.
# %pip install statsmodels

import statsmodels.formula.api as smf

ols_data = survival.copy()
ols_data["question"] = ols_data["question"].astype("category")

ols_model = smf.ols("pupil_left ~ C(question)", data=ols_data).fit(
    cov_type="cluster", cov_kwds={"groups": ols_data["block"]}
)
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:             pupil_left   R-squared:                       0.362
Model:                            OLS   Adj. R-squared:                  0.315
Method:                 Least Squares   F-statistic:                     6.111
Date:                Sat, 21 Nov 2026   Prob (F-statistic):             0.0211
Time:                        08:02:25   Log-Likelihood:                 73.076
No. Observations:                  30   AIC:                            -140.2
Df Residuals:                      27   BIC:                            -135.9
Df Model:                           2                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.0509      0.01